# Craftax tutorial

## Initialization
CrafTax strictly follows the Gymnax API, which means it relies on functional environment transitions. Instead of a stateful `env.step()`, you pass the current state and a PRNG key into a pure function.

In [2]:
import jax
import jax.numpy as jnp
from craftax.craftax_env import make_craftax_env_from_name

# Initialize JAX's pseudo-random number generator (PRNG)
rng = jax.random.PRNGKey(0)

# We will use the Symbolic observation space for faster training/stepping.
# You can also use "Craftax-Pixels-v1" if you want to train CNN-based agents.
env_name = "Craftax-Symbolic-v1"

# Create the environment with auto-reset enabled
env = make_craftax_env_from_name(env_name, auto_reset=True)
env_params = env.default_params

print(f"Environment: {env_name}")
print(f"Action Space size: {env.action_space(env_params).n}")

Loading Craftax textures from cache.
Textures successfully loaded from cache.
Environment: Craftax-Symbolic-v1
Action Space size: 43


## Basic Usage

In [3]:
# Split the RNG key into three new keys for reset, action sampling, and stepping
rng, rng_reset, rng_action, rng_step = jax.random.split(rng, 4)

# 1. Reset the environment to get the initial observation and internal state
obs, state = env.reset(rng_reset, env_params)

print("Initial observation shape:", obs.shape)

# 2. Sample a random action from the action space
# Craftax has a discrete action space (e.g., move, mine, place, craft)
random_action = env.action_space(env_params).sample(rng_action)

# 3. Step the environment forward
next_obs, next_state, reward, done, info = env.step(
    rng_step, 
    state, 
    random_action, 
    env_params
)

print(f"Action taken: {random_action}")
print(f"Reward received: {reward}")
print(f"Episode done: {done}")

Initial observation shape: (8268,)


/home/pablo/VSCodeProjects/open-endedness-project/.venv/lib/python3.12/site-packages/jax/_src/numpy/array_methods.py:125: UserWarning: Explicitly requested dtype int64 requested in astype is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


Action taken: 9
Reward received: 0.0
Episode done: False


## Vectorizing with vmap

In [4]:
# Number of parallel environments
batch_size = 1024

# 1. Create a batch of PRNG keys (one for each environment)
rng, rng_batch_reset = jax.random.split(rng)
batch_reset_keys = jax.random.split(rng_batch_reset, batch_size)

# 2. Vmap the reset function
# We use jax.vmap to map the reset function across our batch of keys.
# The `in_axes=(0, None)` tells JAX to map over the 0th axis of the keys,
# but pass the env_params as a single, unbatched argument.
vmap_reset = jax.vmap(env.reset, in_axes=(0, None))
batch_obs, batch_state = vmap_reset(batch_reset_keys, env_params)

print(f"Batched observation shape: {batch_obs.shape}")
# Expected: (1024, obs_dim)

# 3. Vmap the step function
vmap_step = jax.vmap(env.step, in_axes=(0, 0, 0, None))

# Generate a batch of random actions for our 1024 environments
rng, rng_batch_action = jax.random.split(rng)
batch_action_keys = jax.random.split(rng_batch_action, batch_size)
batch_actions = jax.vmap(env.action_space(env_params).sample)(batch_action_keys)

# Generate a batch of step keys
rng, rng_batch_step = jax.random.split(rng)
batch_step_keys = jax.random.split(rng_batch_step, batch_size)

# Step all 1024 environments simultaneously!
next_batch_obs, next_batch_state, batch_reward, batch_done, batch_info = vmap_step(
    batch_step_keys, batch_state, batch_actions, env_params
)

print(f"Batched reward shape: {batch_reward.shape}")
print(f"Mean reward across batch: {jnp.mean(batch_reward)}")

Batched observation shape: (1024, 8268)
Batched reward shape: (1024,)
Mean reward across batch: 0.0009765625


## Compiling with jit

In [5]:
import time

# JIT compile the vectorized step function
jit_vmap_step = jax.jit(vmap_step)

# Run once to trigger XLA compilation overhead
_ = jit_vmap_step(batch_step_keys, batch_state, batch_actions, env_params)

# Now, let's time it!
start_time = time.time()
steps = 100

for _ in range(steps):
    # (In a real training loop, you would sample new actions from your neural net here)
    rng, rng_step = jax.random.split(rng)
    step_keys = jax.random.split(rng_step, batch_size)
    
    next_batch_obs, next_batch_state, batch_reward, batch_done, batch_info = jit_vmap_step(
        step_keys, 
        batch_state, 
        batch_actions, 
        env_params
    )
    # Update state for the next step
    batch_state = next_batch_state

end_time = time.time()
fps = (batch_size * steps) / (end_time - start_time)

print(f"Successfully ran {batch_size * steps} environment steps in {end_time - start_time:.2f} seconds.")
print(f"Speed: {fps:.0f} Steps Per Second!")

Successfully ran 102400 environment steps in 18.52 seconds.
Speed: 5528 Steps Per Second!
